# CNN Benchmark

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from scipy.io import loadmat
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(42)
np.random.seed(42)

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

device

## Load Dataset

In [ ]:
mat = loadmat("../../data/driver-drowsiness/cui/dataset.mat")

X_cui_full = mat["EEGsample"].astype(np.float32)      # samples x channels x time
y_cui = mat["substate"].reshape(-1).astype(np.int64)
subjects_cui = mat["subindex"].reshape(-1).astype(np.int64)

label_names = {0: "alert", 1: "drowsy"}

# Keep only the channels shared with Orosco: O1, O2, C3, C4.
cui_channel_names = [
    "Fp1", "Fp2", "F7", "F3", "Fz", "F4", "F8",
    "FT7", "FC3", "FCz", "FC4", "FT8",
    "T3", "C3", "Cz", "C4", "T4",
    "TP7", "CP3", "CPz", "CP4", "TP8",
    "T5", "P3", "Pz", "P4", "T6",
    "O1", "Oz", "O2",
]
shared_channels = ["O1", "O2", "C3", "C4"]
cui_shared_indices = [cui_channel_names.index(channel) for channel in shared_channels]
X_cui = X_cui_full[:, cui_shared_indices, :]

print("Cui full X:", X_cui_full.shape)
print("Cui shared-channel X:", X_cui.shape)
print("Cui y:", y_cui.shape, dict(zip(*np.unique(y_cui, return_counts=True))))
print("Cui subjects:", np.unique(subjects_cui).tolist())


## Load Orosco Dataset


In [ ]:
orosco = np.load("../../data/driver-drowsiness/orosco/dataset/orosco_windowed_balanced.npz", allow_pickle=True)

X_orosco = orosco["X"].astype(np.float32)      # samples x channels x time
y_orosco = orosco["Y"].astype(np.int64)
subjects_orosco = orosco["subjects"].astype(np.int64)

print("Orosco X:", X_orosco.shape)
print("Orosco y:", y_orosco.shape, dict(zip(*np.unique(y_orosco, return_counts=True))))
print("Orosco subjects:", np.unique(subjects_orosco).tolist())
print("Orosco channels:", orosco["channels"].tolist())


## Combine Cui and Orosco


In [ ]:
X = np.concatenate([X_cui, X_orosco], axis=0)
y = np.concatenate([y_cui, y_orosco], axis=0)
subjects = np.concatenate([subjects_cui, subjects_orosco], axis=0)
dataset_source = np.concatenate([
    np.full(len(y_cui), "cui"),
    np.full(len(y_orosco), "orosco"),
])

print("Combined X:", X.shape)
print("Combined y:", y.shape, dict(zip(*np.unique(y, return_counts=True))))
print("Combined subjects:", np.unique(subjects).tolist())
print("Dataset counts:", dict(zip(*np.unique(dataset_source, return_counts=True))))
print("Shared channels:", shared_channels)


## Subject-Aware Split

In [ ]:
train_subjects = np.arange(1, 9)   # subjects 1-8
val_subjects = np.array([9])       # subject 9
test_subjects = np.array([10, 11]) # subjects 10-11

train_mask = np.isin(subjects, train_subjects)
val_mask = np.isin(subjects, val_subjects)
test_mask = np.isin(subjects, test_subjects)

X_train, y_train = X[train_mask], y[train_mask]
X_val, y_val = X[val_mask], y[val_mask]
X_test, y_test = X[test_mask], y[test_mask]

# Normalize using training-set statistics only.
channel_mean = X_train.mean(axis=(0, 2), keepdims=True)
channel_std = X_train.std(axis=(0, 2), keepdims=True) + 1e-6

X_train = (X_train - channel_mean) / channel_std
X_val = (X_val - channel_mean) / channel_std
X_test = (X_test - channel_mean) / channel_std

split_summary = pd.DataFrame({
    "split": ["train", "val", "test"],
    "subjects": [train_subjects.tolist(), val_subjects.tolist(), test_subjects.tolist()],
    "samples": [len(y_train), len(y_val), len(y_test)],
    "alert": [(y_train == 0).sum(), (y_val == 0).sum(), (y_test == 0).sum()],
    "drowsy": [(y_train == 1).sum(), (y_val == 1).sum(), (y_test == 1).sum()],
})

display(split_summary)

## DataLoaders

In [ ]:
batch_size = 32

def make_loader(X_array, y_array, shuffle=False):
    X_tensor = torch.tensor(X_array, dtype=torch.float32)
    y_tensor = torch.tensor(y_array, dtype=torch.long)
    dataset = TensorDataset(X_tensor, y_tensor)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)

train_loader = make_loader(X_train, y_train, shuffle=True)
val_loader = make_loader(X_val, y_val)
test_loader = make_loader(X_test, y_test)

xb, yb = next(iter(train_loader))
print("batch X:", xb.shape)  # batch x channels x time
print("batch y:", yb.shape)

## Basic CNN

In [ ]:
class BasicEEGCNN(nn.Module):
    def __init__(self, n_channels=30, n_times=384, n_classes=2):
        super().__init__()

        # Edit this block first when experimenting with CNN designs.
        self.conv = nn.Sequential(
            nn.Conv1d(n_channels, 32, kernel_size=7, padding=3),
            nn.ELU(),
            nn.MaxPool1d(kernel_size=2),

            nn.Conv1d(32, 64, kernel_size=5, padding=2),
            nn.ELU(),
            nn.MaxPool1d(kernel_size=2),

            nn.Conv1d(64, 64, kernel_size=3, padding=1),
            nn.ELU(),
            nn.MaxPool1d(kernel_size=2),

            nn.Flatten(),
        )

        with torch.no_grad():
            dummy_input = torch.zeros(1, n_channels, n_times)
            flattened_size = self.conv(dummy_input).shape[1]

        # Edit this block when experimenting with classifier depth/width.
        self.fc = nn.Sequential(
            nn.Linear(flattened_size, 64),
            nn.ELU(),
            nn.Dropout(p=0.3),
            nn.Linear(64, n_classes),
        )

    def forward(self, x):
        x = self.conv(x)
        return self.fc(x)


model = BasicEEGCNN(n_channels=X_train.shape[1], n_times=X_train.shape[2]).to(device)
model


In [ ]:
def count_parameters(model):
    return sum(param.numel() for param in model.parameters() if param.requires_grad)


with torch.no_grad():
    logits = model(xb.to(device))

print("input batch:", xb.shape)
print("logits:", logits.shape)
print(f"trainable parameters: {count_parameters(model):,}")


## Training Utilities

In [ ]:
def binary_f1_score(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    f1_scores = []
    for label in [0, 1]:
        tp = np.sum((y_true == label) & (y_pred == label))
        fp = np.sum((y_true != label) & (y_pred == label))
        fn = np.sum((y_true == label) & (y_pred != label))

        precision = tp / (tp + fp + 1e-12)
        recall = tp / (tp + fn + 1e-12)
        f1 = 2 * precision * recall / (precision + recall + 1e-12)
        f1_scores.append(f1)

    return float(np.mean(f1_scores))


def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    all_preds = []
    all_targets = []

    with torch.no_grad():
        for batch_x, batch_y in loader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)

            logits = model(batch_x)
            loss = criterion(logits, batch_y)
            preds = logits.argmax(dim=1)

            total_loss += loss.item() * batch_x.size(0)
            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(batch_y.cpu().numpy())

    all_preds = np.array(all_preds)
    all_targets = np.array(all_targets)
    accuracy = float((all_preds == all_targets).mean())
    f1 = binary_f1_score(all_targets, all_preds)
    avg_loss = total_loss / len(loader.dataset)

    return {"loss": avg_loss, "accuracy": accuracy, "f1_macro": f1}


def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss = 0.0

    for batch_x, batch_y in loader:
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)

        optimizer.zero_grad()
        logits = model(batch_x)
        loss = criterion(logits, batch_y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * batch_x.size(0)

    return total_loss / len(loader.dataset)

## Train

In [ ]:
learning_rate = 1e-3
epochs = 20

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-4)

history = []
best_val_f1 = -1.0
best_state = None

for epoch in range(1, epochs + 1):
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer)
    train_metrics = evaluate(model, train_loader, criterion)
    val_metrics = evaluate(model, val_loader, criterion)

    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        "train_accuracy": train_metrics["accuracy"],
        "train_f1": train_metrics["f1_macro"],
        "val_loss": val_metrics["loss"],
        "val_accuracy": val_metrics["accuracy"],
        "val_f1": val_metrics["f1_macro"],
    }
    history.append(row)

    if val_metrics["f1_macro"] > best_val_f1:
        best_val_f1 = val_metrics["f1_macro"]
        best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}

    print(
        f"epoch {epoch:02d} | "
        f"train loss {train_loss:.4f} acc {train_metrics['accuracy']:.3f} f1 {train_metrics['f1_macro']:.3f} | "
        f"val loss {val_metrics['loss']:.4f} acc {val_metrics['accuracy']:.3f} f1 {val_metrics['f1_macro']:.3f}"
    )

history_df = pd.DataFrame(history)
display(history_df.tail())

## Learning Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)

axes[0].plot(history_df["epoch"], history_df["train_loss"], label="train")
axes[0].plot(history_df["epoch"], history_df["val_loss"], label="val")
axes[0].set_title("Loss")
axes[0].set_xlabel("Epoch")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(history_df["epoch"], history_df["train_f1"], label="train")
axes[1].plot(history_df["epoch"], history_df["val_f1"], label="val")
axes[1].set_title("Macro F1")
axes[1].set_xlabel("Epoch")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.show()

## Test Set Evaluation

In [ ]:
if best_state is not None:
    model.load_state_dict(best_state)
    model.to(device)

test_metrics = evaluate(model, test_loader, criterion)
test_metrics